# NB3_closed_loop
gym 안에서 렌더 → 추론 → pure pursuit 폐루프. 먼저 오라클(정답 waypoint + 노이즈)로 지연·노이즈 스윕을 하고, `model.pt`가 있으면 학습 모델로도 달립니다.
결과 파일은 코랩 VM의 `out/`에 생기며 VSCode 탐색기에는 보이지 않습니다. 아래 마지막 셀이 표와 영상을 셀 출력에 직접 띄웁니다.


In [ ]:
REPO_URL = "https://github.com/minjai345/f1tenth_gym.git"   # 조교 fork. 본인 fork를 쓰면 여기만 바꾸세요
BRANCH = "main"
import os
if os.path.isdir("f1tenth_gym"):
    !git -C f1tenth_gym pull -q      # 이미 받아둔 경우 최신 코드로 갱신
else:
    !git clone --branch $BRANCH $REPO_URL
assert os.path.isfile("f1tenth_gym/camsim/requirements.txt"), "clone 실패: REPO_URL/BRANCH 를 확인하세요"
%cd f1tenth_gym
!pip install -q -r camsim/requirements.txt
!bash camsim/scripts/install_gym019.sh
!pip install -q --no-deps -e .
# numpy는 코랩 기본(2.x)을 그대로 쓴다. 혹시 numpy 버전이 바뀌었다는 메시지가 나오면 "런타임 다시 시작" 후 이 셀부터 다시 실행


In [ ]:
!cat camsim/config.yaml


In [ ]:
# 별도 프로세스로 실행: 커널에 남아 있는 예전 camsim 모듈을 재사용하지 않도록 %run 대신 !python 을 쓴다
!python camsim/scripts/nb3_closed_loop.py model.pt


In [ ]:
import json, os, pandas as pd
from IPython.display import Video, display
print('오라클 스윕: 지연(틱) x 노이즈(m) -> 종료 이유 / 평균 횡오차. reason=lap 이 완주')
df = pd.DataFrame(json.load(open('out/sweep_oracle.json')))
display(df.pivot(index='latency_steps', columns='sigma', values='reason'))
display(df.pivot(index='latency_steps', columns='sigma', values='mean_lateral_m').round(3))
if os.path.exists('out/run_latency0.mp4'):
    print('학습 모델 주행 영상 (지연 0). 초록 점 = 모델이 예측한 waypoint')
    display(Video('out/run_latency0.mp4', embed=True, width=640))
else:
    print('영상 없음: model.pt 가 없어 학습 모델 구간을 건너뛰었습니다 (NB2 먼저)')
